# SaShiMi + DiffWave — music generation (single notebook: train → generate any length)

Train **one** model on your clips (a few hours total), then generate audio of **any length** — either **pure generation** (from noise) or **continuation** (extend a clip). Both come from the same model via *overlap outpainting*: it generates a 2 s window, carries the waveform tail as clean context into the next window, and appends only the new samples → **seamless, arbitrary-length** output at 22.05 kHz.

**Runtime → Change runtime type → GPU (L4/A10/A100).** Do *not* pip-install torch/torchaudio (use Colab's CUDA-matched build). ffmpeg is preinstalled.

In [ ]:
# 1. Clone the repo + mount Drive for persistent checkpoints.
!git clone https://github.com/heyuwang1999/diffwave-sashimi.git
%cd diffwave-sashimi
from google.colab import drive; drive.mount('/content/drive')
# vendor/exp is gitignored and /content is wiped on disconnect -> symlink it to Drive so training resumes.
!mkdir -p /content/drive/MyDrive/diffwave-sashimi-exp && ln -sfn /content/drive/MyDrive/diffwave-sashimi-exp vendor/exp

In [ ]:
# 2. GPU check + Python deps (NOT torch/torchaudio).
import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
!pip install -q -r requirements-colab.txt

In [ ]:
# 3. AUTOMATIC PROCESSING: point at your audio on Drive (a FOLDER of clips, or one long file).
#    Decodes any format -> mono 22.05 kHz, streams with bounded memory (a 4-hour file is fine),
#    holds out the last 10% of each track for continuation eval.
AUDIO_SRC = '/content/drive/MyDrive/my_music'   # <-- folder of clips OR a single file (any format)
!python scripts/prepare_data.py --in_dir "{AUDIO_SRC}" --out_dir vendor/data/music --sr 22050 --holdout_frac 0.1

In [ ]:
# 4. (optional) Sanity-check the whole stack on the real GPU (~1-2 min; first run also JIT-compiles S4 kernels).
!cd vendor && python ../scripts/smoke_test.py

In [ ]:
# 5. TRAIN the outpainting model. Resumes automatically from the last checkpoint.
#    batch=4 fits 22-24GB (L4/A10); A100 -> 8; OOM -> 2.
#    A short DDIM monitoring sample is generated at each checkpoint (every 5k iters) — quick, not a hang.
!cd vendor && PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python train.py experiment=music_outpaint train.batch_size_per_gpu=4
# Logging + uploaded samples: add  wandb.mode=online
# 2 h of audio: aim ~100-200k iters; LISTEN at checkpoints and stop before it memorizes.

In [ ]:
# 6a. GENERATE — pure generation, ANY length. Uses fast DDIM by default (set in the experiment).
#     First run prints a one-time "[KeOps] Generating code ... OK" (S4 kernel compile, a few minutes) and
#     then a progress bar — that is NOT a hang. Generation time ~ n_samples x seconds x steps; start small.
!cd vendor && python generate.py experiment=music_outpaint generate.ckpt_iter=max generate.n_samples=2 generate.gen_seconds=15
# Longer / more: generate.gen_seconds=60 generate.n_samples=4   |   even faster: generate.sampling_steps=30

In [ ]:
# 6b. GENERATE — continuation: extend one of your held-out clips by gen_seconds.
import glob
seed = sorted(glob.glob('vendor/data/music_holdout/*.wav'))
print('seed clip:', seed[0] if seed else 'NONE (set --holdout_frac > 0 in step 3)')
if seed:
    ctx = seed[0].replace('vendor/', '')   # path relative to vendor/
    !cd vendor && python generate.py experiment=music_outpaint generate.ckpt_iter=max generate.n_samples=1 generate.gen_seconds=15 generate.context_path="{ctx}"

In [ ]:
# 7. Listen to the latest generated audio.
import glob, IPython.display as ipd
wavs = sorted(glob.glob('vendor/exp/**/waveforms/**/*.wav', recursive=True))
print(wavs[-3:])
ipd.Audio(wavs[-1]) if wavs else print('No samples yet.')

In [ ]:
# 8. Evaluate (A/B): Frechet Mel Distance of generated vs your real audio (lower = closer).
!cd vendor && python ../scripts/evaluate.py fidelity --gen_dir exp/<run>/waveforms/<iter> --ref_dir data/music --sr 22050